In [4]:
!pip install findspark


In [4]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

# Khởi tạo Spark Session
spark = SparkSession.builder.appName("Green").getOrCreate()

print("Spark Session đã được khởi tạo thành công!")


Spark Session đã được khởi tạo thành công!


In [5]:
hvfhv = spark.read.parquet("hdfs://namenode:8020/data/nyc_taxi/fhvhv_tripdata_*.parquet")

In [6]:
hvfhv.show(5)
hvfhv.count()

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+--

239470448

In [7]:
from pyspark.sql.functions import (
    col, to_date, hour, dayofweek, dayofmonth, month, count, when, lag,
    desc, sum as _sum, avg, unix_timestamp
)
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, IndexToString
)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
# BƯỚC 1: LÀM SẠCH VÀ LỌC DỮ LIỆU HVFHV
# ==============================================================================
print("--- BƯỚC 1: Đang làm sạch và lọc dữ liệu HVFHV ---")

# Tính toán tổng tiền (vì không có sẵn)
fare_components = ["base_passenger_fare", "tolls", "bcf", "sales_tax", "congestion_surcharge", "airport_fee", "tips"]
hvfhv_with_fare = hvfhv.na.fill(0, subset=fare_components) \
                       .withColumn("total_amount", 
                                   col("base_passenger_fare") + 
                                   col("tolls") + 
                                   col("bcf") + 
                                   col("sales_tax") + 
                                   col("congestion_surcharge") + 
                                   col("airport_fee") + 
                                   col("tips"))


# Lọc các giá trị null
hvfhv_filtered = hvfhv_with_fare.filter(
    (col("pickup_datetime").isNotNull()) &
    (col("dropoff_datetime").isNotNull()) &
    (col("PULocationID").isNotNull()) &
    (col("DOLocationID").isNotNull())
)

# Lọc các giá trị PULocationID và DOLocationID hợp lệ
hvfhv_filtered = hvfhv_filtered.filter(
    (col("PULocationID") > 0) & (col("PULocationID") <= 263) &
    (col("DOLocationID") > 0) & (col("DOLocationID") <= 263)
)

# Lọc bỏ các chuyến đi có thời gian và quãng đường phi lý
hvfhv_filtered = hvfhv_filtered.withColumn(
    "calculated_trip_time_seconds",
    unix_timestamp(col("dropoff_datetime")) - unix_timestamp(col("pickup_datetime"))
)

hvfhv_filtered = hvfhv_filtered.withColumn(
    "calculated_average_speed_mph",
    (col("trip_miles") / col("calculated_trip_time_seconds")) * 3600
)

hvfhv_filtered = hvfhv_filtered.filter(
    (col("calculated_trip_time_seconds") >= 60) & (col("calculated_trip_time_seconds") <= 18000) &
    (col("trip_miles") >= 0.1) & (col("trip_miles") <= 50) &
    (col("calculated_average_speed_mph") >= 1) & (col("calculated_average_speed_mph") <= 60)
)

# Lọc các giá trị tài chính hợp lệ
hvfhv_filtered = hvfhv_filtered.filter(
    (col("base_passenger_fare") >= 0) & 
    (col("tips") >= 0) &
    (col("tolls") >= 0) & 
    (col("total_amount") > 0) &
    (col("congestion_surcharge") >= 0) & (col("congestion_surcharge") < 20) &
    (col("airport_fee") >= 0)
)

# Gán lại vào biến hvfhv để sử dụng cho các bước sau
hvfhv = hvfhv_filtered
print(f"Tổng số dòng của HVFHV sau khi lọc: {hvfhv.count()}")

--- BƯỚC 1: Đang làm sạch và lọc dữ liệu HVFHV ---
Tổng số dòng của HVFHV sau khi lọc: 229270791


In [9]:
print("\n--- BƯỚC 2: Đang chuẩn bị dữ liệu cho mô hình... ---")

# a. Tổng hợp dữ liệu
hourly_demand = hvfhv.groupBy(
    to_date(col("pickup_datetime")).alias("date"), 
    hour(col("pickup_datetime")).alias("hour"),
    col("PULocationID")
).agg(count("*").alias("trip_count"))

# b. Thêm các đặc trưng thời gian cơ bản
model_data = hourly_demand.withColumn("day_of_week", dayofweek(col("date"))) \
                          .withColumn("day_of_month", dayofmonth(col("date"))) \
                          .withColumn("month", month(col("date")))

# c. Thêm đặc trưng Kỳ nghỉ
holiday_list = [("2024-01-01", "Holiday"), ("2024-01-15", "Holiday"), ("2024-05-27", "Holiday"), 
                ("2024-07-04", "Holiday"), ("2024-09-02", "Holiday"), ("2024-11-28", "Holiday"), 
                ("2024-12-25", "Holiday")]
holiday_periods_list = []
for holiday_date_str, type in holiday_list:
    holiday_date = pd.to_datetime(holiday_date_str)
    holiday_periods_list.append((str(holiday_date.date()), type))
    holiday_periods_list.append((str((holiday_date - pd.DateOffset(days=1)).date()), "Pre-Holiday"))
    holiday_periods_list.append((str((holiday_date + pd.DateOffset(days=1)).date()), "Post-Holiday"))

holiday_periods_df = spark.createDataFrame(holiday_periods_list, ["date_str", "day_type"]) \
                          .withColumn("date", to_date(col("date_str"))).drop("date_str")

model_data_with_holidays = model_data.join(holiday_periods_df, ["date"], "left") \
                                     .na.fill({"day_type": "Normal_Day"})

# d. Thêm đặc trưng Trễ
window_spec = Window.partitionBy("PULocationID").orderBy("date", "hour")
data_with_features = model_data_with_holidays.withColumn("lag_24hr", lag("trip_count", 24).over(window_spec)) \
                                             .withColumn("lag_168hr", lag("trip_count", 24 * 7).over(window_spec)) \
                                             .na.fill(0, subset=["lag_24hr", "lag_168hr"])

# e. Tạo nhãn phân loại và trọng số
quantiles = data_with_features.approxQuantile("trip_count", [0.25, 0.5, 0.75, 0.95], 0.01)
data_with_levels = data_with_features.withColumn("demand_level",
    when(col("trip_count") == 0, "None")
    .when(col("trip_count") <= quantiles[0], "Very_Low")
    .when(col("trip_count") <= quantiles[1], "Low")
    .when(col("trip_count") <= quantiles[2], "Medium")
    .when(col("trip_count") <= quantiles[3], "High")
    .otherwise("Very_High")
)
class_counts = data_with_levels.groupBy("demand_level").count()
total_samples = data_with_levels.count()
num_classes = class_counts.count()
calculate_weight = class_counts.withColumn("weight", total_samples / (num_classes * col("count")))
final_data_for_pipeline = data_with_levels.join(calculate_weight.select("demand_level", "weight"), ["demand_level"], "left")



--- BƯỚC 2: Đang chuẩn bị dữ liệu cho mô hình... ---


In [10]:
# ==============================================================================
print("\n--- BƯỚC 3: Bắt đầu xây dựng, huấn luyện và đánh giá Pipeline... ---")

# a. Định nghĩa các bước trong Pipeline
label_indexer = StringIndexer(inputCol="demand_level", outputCol="label", handleInvalid="keep")
day_type_indexer = StringIndexer(inputCol="day_type", outputCol="day_type_index", handleInvalid="keep")
day_type_encoder = OneHotEncoder(inputCol="day_type_index", outputCol="day_type_vec")
feature_columns = ["PULocationID", "hour", "day_of_week", "day_of_month", "month", "lag_24hr","lag_168hr", "day_type_vec"]
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features", handleInvalid="skip")
rf_classifier = RandomForestClassifier(featuresCol="features", labelCol="label", weightCol="weight", seed=42)
pipeline = Pipeline(stages=[label_indexer, day_type_indexer, day_type_encoder, assembler, rf_classifier])

# b. Chia dữ liệu Train/Test
train_data, test_data = final_data_for_pipeline.randomSplit([0.8, 0.2], seed=42)

# c. Huấn luyện Pipeline
pipeline_model = pipeline.fit(train_data)
print("Đã huấn luyện xong Pipeline!")

# d. Thực hiện dự đoán
predictions = pipeline_model.transform(test_data)

# e. Đánh giá mô hình
print("\n--- Bảng kết quả đánh giá mô hình ---")
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
accuracy = evaluator_accuracy.evaluate(predictions)
f1_score = evaluator_f1.evaluate(predictions)
print(f"Độ chính xác (Accuracy): {accuracy * 100:.2f}%")
print(f"Chỉ số F1-Score (trung bình): {f1_score:.2f}")

# f. Hiển thị kết quả chi tiết
print("\n--- Ma trận Nhầm lẫn (Confusion Matrix) ---")
labels = pipeline_model.stages[0].labels
prediction_converter = IndexToString(inputCol="prediction", outputCol="predicted_level", labels=labels)
final_output = prediction_converter.transform(predictions)
final_output.groupBy("demand_level").pivot("predicted_level", labels).count().na.fill(0).show()










--- BƯỚC 3: Bắt đầu xây dựng, huấn luyện và đánh giá Pipeline... ---
Đã huấn luyện xong Pipeline!

--- Bảng kết quả đánh giá mô hình ---
Độ chính xác (Accuracy): 79.46%
Chỉ số F1-Score (trung bình): 0.80

--- Ma trận Nhầm lẫn (Confusion Matrix) ---
+------------+--------+------+-----+-----+---------+
|demand_level|Very_Low|Medium|  Low| High|Very_High|
+------------+--------+------+-----+-----+---------+
|        High|     242| 10340|  538|66618|     8735|
|         Low|    9655| 12248|86007|  313|       54|
|   Very_High|      79|   292|   23| 3518|    20594|
|      Medium|     656| 79886|17262|12234|      256|
|    Very_Low|   98602|   207|13616|   95|       44|
+------------+--------+------+-----+-----+---------+



In [11]:
from pyspark.sql.functions import lit, to_date, dayofweek, dayofmonth, month, col
from pyspark.ml.feature import IndexToString
import pandas as pd

def get_hvfhv_prediction(location_id, date_str, hour_of_day):
    """
    Hàm này nhận đầu vào là địa điểm, ngày, giờ và trả về dự đoán mức độ nhu cầu
    dựa trên mô hình RandomForest duy nhất đã được huấn luyện.
    (Phiên bản cập nhật với lag_168hr)
    """
    print(f"--- Đang tạo dự đoán cho LocationID: {location_id}, Thời gian: {hour_of_day}h ngày {date_str} ---")

    # ----- 1. Chuẩn bị DataFrame đầu vào cho mô hình -----
    try:
        # a. Tạo DataFrame một dòng cho dữ liệu đầu vào
        input_data = spark.createDataFrame([(location_id, date_str, hour_of_day)], ["PULocationID", "date_str", "hour"])
        input_data = input_data.withColumn("date", to_date(col("date_str"))) \
                               .withColumn("day_of_week", dayofweek(col("date"))) \
                               .withColumn("day_of_month", dayofmonth(col("date"))) \
                               .withColumn("month", month(col("date")))

        # b. Thêm feature kỳ nghỉ
        input_data = input_data.join(holiday_periods_df, ["date"], "left").na.fill({"day_type": "Normal_Day"})

        # c. Thêm feature trễ (lag_24hr) bằng cách tra cứu dữ liệu lịch sử
        previous_day_date = pd.to_datetime(date_str) - pd.DateOffset(days=1)
        previous_day_str = previous_day_date.strftime('%Y-%m-%d')
        
        lag_24hr_row = hourly_demand.filter(
            (col("PULocationID") == location_id) &
            (to_date(col("date")) == lit(previous_day_str)) &
            (col("hour") == hour_of_day)
        ).first()
        
        lag_24hr_value = lag_24hr_row["trip_count"] if lag_24hr_row else 0
        input_data = input_data.withColumn("lag_24hr", lit(lag_24hr_value))

        # ======> THÊM MỚI: Tạo đặc trưng lag_168hr <======
        previous_week_date = pd.to_datetime(date_str) - pd.DateOffset(days=7)
        previous_week_str = previous_week_date.strftime('%Y-%m-%d')
        
        lag_168hr_row = hourly_demand.filter(
            (col("PULocationID") == location_id) &
            (to_date(col("date")) == lit(previous_week_str)) &
            (col("hour") == hour_of_day)
        ).first()
        
        lag_168hr_value = lag_168hr_row["trip_count"] if lag_168hr_row else 0
        input_data = input_data.withColumn("lag_168hr", lit(lag_168hr_value))


    except Exception as e:
        return f"Lỗi khi chuẩn bị feature cho mô hình: {e}"

    # ----- 2. Áp dụng mô hình và trả về kết quả -----
    try:
        # a. Sử dụng pipeline_model đã huấn luyện để dự đoán
        prediction_result = pipeline_model.transform(input_data)

        # b. Chuyển đổi nhãn dự đoán (dạng số) về lại dạng chữ
        labels = pipeline_model.stages[0].labels 
        prediction_converter = IndexToString(inputCol="prediction", outputCol="predicted_level", labels=labels)
        
        predicted_level = prediction_converter.transform(prediction_result).first()["predicted_level"]
        
        return f"Dự đoán: Nhu cầu tại LocationID {location_id} vào {hour_of_day}h ngày {date_str} sẽ ở mức '{predicted_level}'."

    except Exception as e:
        return f"Lỗi khi thực hiện dự đoán: {e}"

In [16]:
# Ví dụ: Kiểm tra nhu cầu cho một địa điểm cụ thể vào một thời điểm trong tương lai
location = 122
date = "2025-01-01" # Một ngày thứ Hai
hour = 18 # 6 giờ tối

# Gọi hàm
recommendation = get_hvfhv_prediction(location_id=location, date_str=date, hour_of_day=hour)

# In kết quả
print("\n==> KẾT QUẢ TRUY VẤN:")
print(recommendation)

--- Đang tạo dự đoán cho LocationID: 122, Thời gian: 18h ngày 2025-01-01 ---

==> KẾT QUẢ TRUY VẤN:
Dự đoán: Nhu cầu tại LocationID 122 vào 18h ngày 2025-01-01 sẽ ở mức 'Low'.


In [30]:
from pyspark.sql.functions import lit, to_date, dayofweek, dayofmonth, month, col, broadcast
from pyspark.sql.types import IntegerType

print("--- BẮT ĐẦU DỰ ĐOÁN CHO NGÀY 1/1/2025 TRÊN TOÀN NYC ---")

# --- Bước 1: Tạo lưới dữ liệu (Grid) cho ngày dự đoán ---
# Lưới này bao gồm tất cả các địa điểm (PULocationID) và tất cả các giờ trong ngày.

# Tạo DataFrame chứa tất cả các giờ (0-23)
hours_df = spark.range(0, 24).withColumnRenamed("id", "hour")

# Tạo DataFrame chứa tất cả các địa điểm (1-263)
locations_df = spark.range(1, 264).withColumnRenamed("id", "PULocationID").withColumn("PULocationID", col("PULocationID").cast(IntegerType()))

# Sử dụng Cross Join để tạo mọi tổ hợp (giờ, địa điểm)
prediction_grid = locations_df.crossJoin(broadcast(hours_df))

# --- Bước 2: Thêm các đặc trưng về thời gian và loại ngày ---
target_date = "2025-01-01"
prediction_grid = prediction_grid.withColumn("date", to_date(lit(target_date))) \
                                 .withColumn("day_of_week", dayofweek(col("date"))) \
                                 .withColumn("day_of_month", dayofmonth(col("date"))) \
                                 .withColumn("month", month(col("date"))) \
                                 .withColumn("day_type", lit("Holiday")) # 1/1 là ngày lễ

# --- Bước 3: Lấy và kết hợp dữ liệu quá khứ (Lag Features) ---
# Chúng ta cần nhu cầu của cùng giờ/địa điểm từ ngày hôm trước (lag_24hr) và tuần trước (lag_168hr).
# Dữ liệu này được lấy từ DataFrame hourly_demand đã tính toán ở bước [9].

# Dữ liệu lag 24 giờ (dữ liệu của ngày 2024-12-31)
lag_24hr_data = hourly_demand.filter(col("date") == "2024-12-31") \
                             .select(
                                 col("PULocationID"), 
                                 col("hour"), 
                                 col("trip_count").alias("lag_24hr")
                             )

# Dữ liệu lag 168 giờ (dữ liệu của ngày 2024-12-25)
lag_168hr_data = hourly_demand.filter(col("date") == "2024-12-25") \
                              .select(
                                  col("PULocationID"), 
                                  col("hour"), 
                                  col("trip_count").alias("lag_168hr")
                              )

# Kết hợp dữ liệu lag vào lưới dự đoán. Sử dụng left join và điền 0 cho các giá trị thiếu.
future_data_to_predict = prediction_grid.join(lag_24hr_data, ["PULocationID", "hour"], "left") \
                                          .join(lag_168hr_data, ["PULocationID", "hour"], "left") \
                                          .na.fill(0, subset=["lag_24hr", "lag_168hr"])

print(f"Đã tạo {future_data_to_predict.count()} điểm dữ liệu để dự đoán cho ngày {target_date}.")
print("Dữ liệu đầu vào mẫu:")
future_data_to_predict.show(5)

# --- Bước 4: Áp dụng Pipeline để dự đoán ---
# Sử dụng pipeline_model đã huấn luyện từ cách 2 để biến đổi dữ liệu mới và đưa ra dự đoán.
# Pipeline sẽ tự động xử lý việc chuyển đổi 'day_type' và lắp ráp các features.
future_predictions_raw = pipeline_model.transform(future_data_to_predict)


# --- Bước 5: Chuyển đổi nhãn dự đoán từ số sang chữ ---
# Sử dụng lại bộ chuyển đổi IndexToString như đã làm trước đó.
label_converter = IndexToString(
    inputCol="prediction",
    outputCol="predicted_level",
    labels=pipeline_model.stages[0].labels
)
final_future_predictions = label_converter.transform(future_predictions_raw)


# --- Bước 6: Hiển thị kết quả dự đoán ---
print(f"\n--- KẾT QUẢ DỰ ĐOÁN NHU CẦU CHO NGÀY {target_date} ---")

# Xem ví dụ dự đoán cho Sân bay JFK (PULocationID = 132) vào các giờ cao điểm
print("\n>>> Dự đoán cho Sân bay JFK (ID=132):")
final_future_predictions.filter(col("PULocationID") == 132) \
                        .select("hour", "predicted_level", "lag_24hr", "lag_168hr") \
                        .orderBy("hour") \
                        .show()

# Thống kê tổng quan về mức nhu cầu được dự đoán trên toàn thành phố
print("\n>>> Tổng quan dự đoán trên toàn thành phố:")
final_future_predictions.groupBy("predicted_level").count().orderBy(col("count").desc()).show()


# --- Bước 7: Xuất kết quả dự đoán ra file CSV ---

print("\n--- Đang xuất kết quả dự đoán ra file CSV ---")

# Chọn các cột cần thiết để xuất
output_df = final_future_predictions.select(
    "PULocationID", 
    "hour", 
    "date", 
    "day_of_week", 
    "predicted_level"
).orderBy("PULocationID", "hour")

# Ghi ra file CSV
# .coalesce(1) -> Gom dữ liệu vào 1 partition để xuất ra 1 file duy nhất.
# .option("header", "true") -> Thêm dòng tiêu đề cho các cột.
# .mode("overwrite") -> Nếu thư mục đã tồn tại, ghi đè lên nó.


--- BẮT ĐẦU DỰ ĐOÁN CHO NGÀY 1/1/2025 TRÊN TOÀN NYC ---
Đã tạo 6312 điểm dữ liệu để dự đoán cho ngày 2025-01-01.
Dữ liệu đầu vào mẫu:
+------------+----+----------+-----------+------------+-----+--------+--------+---------+
|PULocationID|hour|      date|day_of_week|day_of_month|month|day_type|lag_24hr|lag_168hr|
+------------+----+----------+-----------+------------+-----+--------+--------+---------+
|           1|   0|2025-01-01|          4|           1|    1| Holiday|       0|        0|
|           1|   3|2025-01-01|          4|           1|    1| Holiday|       0|        0|
|           1|   5|2025-01-01|          4|           1|    1| Holiday|       0|        0|
|           1|   4|2025-01-01|          4|           1|    1| Holiday|       0|        0|
|           1|   1|2025-01-01|          4|           1|    1| Holiday|       0|        0|
+------------+----+----------+-----------+------------+-----+--------+--------+---------+
only showing top 5 rows


--- KẾT QUẢ DỰ ĐOÁN NHU CẦU CH

In [ ]:
output_path_hdfs = "hdfs://namenode:8020/data/output/predictions_2025_01_01"

# Ghi dữ liệu vào HDFS
output_df.coalesce(1).write \
    .option("header", "true") \
    .mode("overwrite") \
    .csv(output_path_hdfs)

print("✅ Đã ghi thành công file CSV vào HDFS tại:", output_path_hdfs)

In [29]:
# Đọc từ HDFS
df = spark.read.csv("hdfs://namenode:8020/data/output/predictions_2025_01_01", header=True)

# Ghi về local (trong container Jupyter)
df.coalesce(1).write.option("header", "true").mode("overwrite").csv("/home/jovyan/predictions_local")
